In [1]:
#Import Libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
#Load the Dataset

In [4]:
dataset = pd.read_csv('Mall_Customers.csv')

In [3]:
#Display the dataset

In [6]:
dataset

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40
...,...,...,...,...,...
195,196,Female,35,120,79
196,197,Female,45,126,28
197,198,Male,32,126,74
198,199,Male,32,137,18


In [4]:
#Select annual income and spending score

In [8]:
# Select all customers and the two clustering features
X = dataset.iloc[:, [3, 4]].values

print("Shape:", X.shape)
print("First five customers:")
print(X[:5])

Shape: (200, 2)
First five customers:
[[15 39]
 [15 81]
 [16  6]
 [16 77]
 [17 40]]


In [5]:
#Scale the features

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Shape:", X_scaled.shape)
print("First five scaled rows:")
print(X_scaled[:5])

Shape: (200, 2)
First five scaled rows:
[[-1.73899919 -0.43480148]
 [-1.73899919  1.19570407]
 [-1.70082976 -1.71591298]
 [-1.70082976  1.04041783]
 [-1.66266033 -0.39597992]]


In [6]:
#Create and train the OPTICS model

In [12]:
# `threshold=0.3` | Limits the radius of a subcluster when adding a customer; smaller values generally create more subclusters 
# `branching_factor=50` | Maximum number of subclusters stored within each tree node
# `n_clusters=5` | Requests five final clusters 
# `compute_labels=True` | Calculates a label for every customer 
#With n_clusters=None, BIRCH returns its subclusters without the final merging step. 
#If fewer subclusters exist than the requested final clusters, it cannot produce the requested number; reducing threshold can help

In [13]:
from sklearn.cluster import Birch

# Starting settings for practice
BR = Birch (threshold=0.3, branching_factor=50, n_clusters=5, compute_labels=True)

# Train once and obtain the cluster labels
y_BR = BR.fit_predict(X_scaled)


In [7]:
#Display the labels
#Labels such as 0, 1, and 2 identify groups, not rankings.

In [15]:
print("Cluster labels:")
print(y_BR)

print("\nUnique labels: ")
print(np.unique(y_BR))

Cluster labels:
[0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0 3 0
 3 0 3 0 3 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1
 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2 1 2
 1 2 1 2 1 2 1 2 1 2 4 2 4 2 4]

Unique labels: 
[0 1 2 3 4]


In [8]:
#Count clusters and noise correctly
#The initial subcluster count can be larger than the final cluster count.

In [17]:
cluster_ids = np.unique(y_BR)

n_clusters = len(cluster_ids)
n_subclusters = len(BR.subcluster_centers_)

print("Number of final clusters:", n_clusters)
print("Number of initial subclusters:", n_subclusters)
print("Total customers:", len(y_BR))

Number of final clusters: 5
Number of initial subclusters: 23
Total customers: 200


In [9]:
#Add the results to your dataset

In [19]:
dataset["BIRCH_Cluster"] = y_BR

print("Customers:", len(dataset))
print("Cluster labels:", len(y_BR))

display(dataset.head(10))

Customers: 200
Cluster labels: 200


,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100),BIRCH_Cluster
0,1,Male,19,15,39,0
1,2,Male,21,15,81,3
2,3,Female,20,16,6,0
3,4,Female,23,16,77,3
4,5,Female,31,17,40,0
5,6,Female,22,17,76,3
6,7,Female,35,18,6,0
7,8,Female,23,18,94,3
8,9,Male,64,19,3,0
9,10,Female,30,19,72,3


In [10]:
#Summarize each group
#Use this table to understand each group’s size and average income and spending.

In [21]:
summary = dataset.groupby("BIRCH_Cluster").agg(
    Customers=("CustomerID", "count"),
    Average_Income=("Annual Income (k$)", "mean"),
    Average_Spending=("Spending Score (1-100)", "mean")
).round(2)

display(summary)

,Customers,Average_Income,Average_Spending
BIRCH_Cluster,,,
0,101,48.17,43.40
1,36,83.11,82.42
2,38,87.00,18.63
3,22,25.73,79.36
4,3,127.67,78.67


In [47]:
#Inspect the selected clusters

In [22]:
# Change this value to inspect another cluster
selected_cluster = 1

cluster_customers = dataset[
    dataset["BIRCH_Cluster"] == selected_cluster
].copy()

display(cluster_customers)

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100),BIRCH_Cluster
123,124,Male,39,69,91,1
125,126,Female,31,70,77,1
127,128,Male,40,71,95,1
129,130,Male,38,71,75,1
131,132,Male,39,71,75,1
133,134,Female,31,72,71,1
135,136,Female,29,73,88,1
137,138,Male,32,73,73,1
139,140,Female,35,74,72,1
141,142,Male,32,75,93,1


In [11]:
#Calculate the silhouette score
#All customers are included in this calculation.

In [25]:
from sklearn.metrics import silhouette_score

if 2 <= n_clusters < len(X_scaled):

    score = silhouette_score(
        X_scaled,
        y_BR
    )

    print("Silhouette score:", round(score, 4))

else:
    print(
        "Silhouette score requires at least two clusters "
        "and more customers than clusters."
    )

Silhouette score: 0.4781
